In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- document_info_tolist ---
FIX_DOCUMENT_INFO_TOLIST_PD = pd.DataFrame({
    "document_id": [1, 2, 3],
    "title": ["Paper A", "Paper B", "Paper C"],
    "abstract": ["Abstract 1", "Abstract 2", "Abstract 3"],
})
FIX_DOCUMENT_INFO_TOLIST_PL = pl.from_pandas(FIX_DOCUMENT_INFO_TOLIST_PD)

df = FIX_DOCUMENT_INFO_TOLIST_PD
print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_document_info_tolist():
    document_ids = df["document_id"].tolist()
    titles = df["title"].tolist()
    abstracts = df["abstract"].tolist()
    return document_ids, titles, abstracts


In [ ]:
# ── Generated wrappers (experiment-generated Polars) ─────────────────────────

def gen_document_info_tolist():

    document_ids = df["document_id"].to_list()
    titles = df["title"].to_list()
    abstracts = df["abstract"].to_list()
    return document_ids, titles, abstracts


In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: document_info_tolist ===
def _run_document_info(frame, func):
    global df; _old=df
    try: df=frame; return func()
    finally: df=_old
try:
    _r=_run_document_info(FIX_DOCUMENT_INFO_TOLIST_PL,gen_document_info_tolist); print("✅ L1 smoke gen_document_info_tolist: OK, type=",type(_r).__name__)
except Exception as _e: print(f"❌ L1 smoke gen_document_info_tolist: {type(_e).__name__}: {_e}")
try:
    _rb=_run_document_info(FIX_DOCUMENT_INFO_TOLIST_PD,before_document_info_tolist); print("✅ L1 smoke before_document_info_tolist: OK")
except Exception as _e: print(f"❌ L1 smoke before_document_info_tolist: {type(_e).__name__}: {_e}")
try:
    _rb=_run_document_info(FIX_DOCUMENT_INFO_TOLIST_PD,before_document_info_tolist); _rg=_run_document_info(FIX_DOCUMENT_INFO_TOLIST_PL,gen_document_info_tolist)
    if _rb==_rg: print("✅ L2 equivalence document_info_tolist all lists: MATCH")
    else: print(f"❌ L2 equivalence document_info_tolist all lists: MISMATCH — before={_rb!r}, gen={_rg!r}")
except Exception as _e: print(f"❌ L2 equivalence document_info_tolist: setup error — {type(_e).__name__}: {_e}")
try:
    _rb=_run_document_info(FIX_DOCUMENT_INFO_TOLIST_PD.head(0),before_document_info_tolist); _rg=_run_document_info(FIX_DOCUMENT_INFO_TOLIST_PL.head(0),gen_document_info_tolist)
    if _rb==_rg==([],[],[]): print("✅ L3 edge document_info_tolist empty all lists: MATCH")
    else: print(f"❌ L3 edge document_info_tolist empty all lists: MISMATCH — before={_rb!r}, gen={_rg!r}")
except Exception as _e: print(f"❌ L3 edge document_info_tolist: {type(_e).__name__}: {_e}")
